# 03 - Modélisation et tracking MLflow

Objectifs de ce notebook :
- Mettre en place le tracking d'expérimentations avec MLflow (étape 2)
- Entraîner et comparer plusieurs modèles avec leurs hyperparamètres par défaut (étape 3)

Modèles testés :
- **Logistic Regression** : modèle simple, baseline
- **Random Forest** : famille des forêts aléatoires
- - **XGBoost** : famille du boosting, référence historique (réputé efficace sur ce type de données)
- **LightGBM** : famille du boosting, plus récent et souvent plus rapide

Métriques calculées :
- AUC (métrique technique principale, peu sensible au déséquilibre)
- Accuracy, Precision, Recall, F1
- **Score métier custom** : coût pondéré où un faux négatif (FN) coûte 10× plus qu'un faux positif (FP), au seuil 0.5

L'optimisation des hyperparamètres et du seuil de décision est réservée à l'étape suivante (notebook 04).

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, make_scorer
)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../data/processed/dataset.parquet'
RANDOM_STATE = 42

## 2. Chargement du dataset

On charge le dataset enrichi produit par le notebook 02. On sépare la cible `TARGET` et on enlève l'identifiant client `SK_ID_CURR` qui ne doit pas servir de feature.

In [2]:
df = pd.read_parquet(DATA_PATH)
print(f"Shape : {df.shape}")

y = df['TARGET']
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])

# Remplacer les valeurs infinies par NaN (issues de divisions par zéro dans les ratios calculés au notebook 02). L'imputer du Pipeline les gérera ensuite.
X = X.replace([np.inf, -np.inf], np.nan)

# LightGBM n'accepte pas les caractères spéciaux dans les noms de colonnes, on remplace par des underscores
X.columns = [c.replace(' ', '_').replace(',', '_').replace(':', '_') for c in X.columns]

print(f"X : {X.shape}")
print(f"y : {y.shape}, distribution : {y.value_counts(normalize=True).round(3).to_dict()}")

Shape : (307507, 797)
X : (307507, 795)
y : (307507,), distribution : {0: 0.919, 1: 0.081}


## 3. Train/test split stratifié

On garde 20% pour le test final. La stratification (`stratify=y`) assure que le split conserve la proportion 92/8 dans le train ET dans le test.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train : {X_train.shape}, distribution TARGET : {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Test  : {X_test.shape}, distribution TARGET : {y_test.value_counts(normalize=True).round(3).to_dict()}")

Train : (246005, 795), distribution TARGET : {0: 0.919, 1: 0.081}
Test  : (61502, 795), distribution TARGET : {0: 0.919, 1: 0.081}


## 4. Score métier custom

Dans le contexte crédit, les erreurs n'ont pas le même coût :
- **Faux négatif (FN)** : on prédit "bon client" alors qu'il fait défaut → crédit accordé, perte du capital prêté. Coût élevé.
- **Faux positif (FP)** : on prédit "mauvais client" alors qu'il aurait remboursé → crédit refusé, manque à gagner sur la marge. Coût faible.

Comme indiqué dans le brief, on suppose que **le coût d'un FN est 10× supérieur à celui d'un FP**.

On définit donc une fonction `business_cost` qui calcule le coût total des erreurs. Plus c'est bas, mieux c'est.

Pour pouvoir l'utiliser en cross-validation, on en fait aussi un "scorer" sklearn (en utilisant `make_scorer` avec `greater_is_better=False` puisque c'est un coût à minimiser).

In [4]:
def business_cost(y_true, y_pred, fn_cost=10, fp_cost=1):
    """
    Coût métier total : FN_count * fn_cost + FP_count * fp_cost.
    Plus c'est bas, mieux c'est.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fn * fn_cost + fp * fp_cost

# Scorer sklearn pour la cross-validation (greater_is_better=False car c'est un coût)
business_cost_scorer = make_scorer(business_cost, greater_is_better=False)

## 5. Setup MLflow

MLflow est configuré en local : tous les runs sont stockés dans le dossier `mlruns/` à la racine du projet. Pour visualiser les résultats, on lancera `mlflow ui` dans un terminal séparé (voir dernière section du notebook).

On définit un nom d'experiment qui regroupera tous nos runs, et on active **`mlflow.sklearn.autolog()`** pour récupérer automatiquement les hyperparamètres des modèles. On loguera en plus manuellement les métriques métier custom (cross-validation, business cost, métriques sur le test) qui ne sont pas couvertes par l'autolog.

In [5]:
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('P6_credit_scoring')

# Autolog pour les params et le modèle. On garde log_models=False pour éviter le doublon avec notre log_model manuel plus bas, qui contrôle le nom de l'artefact.
mlflow.sklearn.autolog(log_models=False)

## 6. Pipelines des modèles

On définit pour chaque modèle un Pipeline scikit-learn qui enchaîne :
1. **Imputation** : on remplit les NaN par la médiane (cf. notebook 01). On le fait dans le Pipeline pour éviter le data leakage en cross-validation.
2. **Scaling** : standardisation pour la régression logistique (qui en a besoin). Random Forest, XGBoost et LightGBM s'en passent (les arbres sont insensibles à l'échelle), on le retire de leurs pipelines.
3. **Modèle** : avec gestion du déséquilibre (`class_weight='balanced'` ou `scale_pos_weight` selon le modèle).

Tous les modèles utilisent leurs hyperparamètres par défaut à cette étape (sauf ceux nécessaires pour la gestion du déséquilibre et la reproductibilité). L'optimisation des hyperparamètres se fera à l'étape suivante.

Pour `scale_pos_weight` (LightGBM et XGBoost) : c'est l'équivalent de `class_weight='balanced'` mais spécifique à ces deux modèles. La valeur recommandée est `n_samples_negatif / n_samples_positif`, soit environ 11 dans notre cas (92 / 8).

In [6]:
# Calcul du ratio pour scale_pos_weight de LightGBM
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight pour LightGBM : {scale_pos_weight:.2f}")

models = {
    'LogisticRegression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=RANDOM_STATE,
        )),
    ]),
    'RandomForest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        )),
    ]),
    'LightGBM': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', LGBMClassifier(
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1
        )),
    ]),
    'XGBoost': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            eval_metric='auc',  # évite un warning
        )),
    ]),
}

print(f"\nModèles prêts : {list(models.keys())}")

scale_pos_weight pour LightGBM : 11.39

Modèles prêts : ['LogisticRegression', 'RandomForest', 'LightGBM', 'XGBoost']


## 7. Entraînement avec cross-validation et tracking MLflow

Pour chaque modèle :
1. On lance une **cross-validation stratifiée à 5 folds** sur le train pour estimer la robustesse des métriques.
2. On entraîne le modèle sur le train complet.
3. On évalue sur le test.
4. On logue tout dans MLflow : params, métriques (CV moyenne+écart-type, et test), modèle, tags.

La cross-validation peut prendre plusieurs minutes (surtout pour Random Forest).

In [7]:
# StratifiedKFold pour garder la proportion 92/8 dans chaque fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Scorers pour la cross-validation
scorers = {
    'auc': 'roc_auc',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'business_cost': business_cost_scorer,
}

results = {}

for model_name, pipeline in models.items():
    print(f"\n{'='*60}")
    print(f"Entraînement : {model_name}")
    print(f"{'='*60}")

    with mlflow.start_run(run_name=model_name):
        # 1. Cross-validation sur le train
        print("Cross-validation 5 folds...")
        cv_results = cross_validate(
            pipeline, X_train, y_train,
            cv=cv,
            scoring=scorers,
            n_jobs=-1,
            return_train_score=False,
        )

        # 2. Entraînement final sur tout le train
        print("Entraînement final sur le train complet...")
        pipeline.fit(X_train, y_train)

        # 3. Évaluation sur le test
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        test_metrics = {
            'test_auc': roc_auc_score(y_test, y_proba),
            'test_accuracy': accuracy_score(y_test, y_pred),
            'test_precision': precision_score(y_test, y_pred),
            'test_recall': recall_score(y_test, y_pred),
            'test_f1': f1_score(y_test, y_pred),
            'test_business_cost': business_cost(y_test, y_pred),
        }

        # 4. Logging MLflow
        # Tags pour s'y retrouver dans l'UI
        mlflow.set_tag('model_family', model_name)
        mlflow.set_tag('imbalance_strategy', 'class_weight_balanced')
        mlflow.set_tag('hyperparams', 'default')

        # Params : nom du modèle + ses hyperparamètres principaux
        model_params = pipeline.named_steps['model'].get_params()
        # On log que quelques params clés, sinon ça pollue l'UI
        params_to_log = {k: v for k, v in model_params.items() 
                        if k in ['random_state', 'class_weight', 'scale_pos_weight', 
                                'n_estimators', 'max_iter', 'max_depth']}
        mlflow.log_params(params_to_log)
        mlflow.log_param('n_train_samples', len(X_train))
        mlflow.log_param('n_features', X_train.shape[1])

        # Métriques de cross-validation (moyenne + écart-type)
        # Note : business_cost est négatif car on a utilisé greater_is_better=False, donc on remet en positif pour la lisibilité.
        for metric in ['auc', 'accuracy', 'precision', 'recall', 'f1']:
            scores = cv_results[f'test_{metric}']
            mlflow.log_metric(f'cv_{metric}_mean', scores.mean())
            mlflow.log_metric(f'cv_{metric}_std', scores.std())
        
        bc_scores = -cv_results['test_business_cost']
        mlflow.log_metric('cv_business_cost_mean', bc_scores.mean())
        mlflow.log_metric('cv_business_cost_std', bc_scores.std())

        # Métriques sur le test final
        for k, v in test_metrics.items():
            mlflow.log_metric(k, v)

        # Modèle sérialisé
        mlflow.sklearn.log_model(pipeline, name='model')

        # On garde une trace en mémoire pour le récap
        results[model_name] = {
            'cv_auc_mean': cv_results['test_auc'].mean(),
            'cv_auc_std': cv_results['test_auc'].std(),
            'cv_business_cost_mean': bc_scores.mean(),
            **test_metrics,
        }

        print(f"CV AUC      : {results[model_name]['cv_auc_mean']:.4f} (± {results[model_name]['cv_auc_std']:.4f})")
        print(f"Test AUC    : {test_metrics['test_auc']:.4f}")
        print(f"Test recall : {test_metrics['test_recall']:.4f}")
        print(f"Test cost   : {test_metrics['test_business_cost']:.0f}")


Entraînement : LogisticRegression
Cross-validation 5 folds...
Entraînement final sur le train complet...


2026/05/11 17:47:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/11 17:48:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


CV AUC      : 0.7708 (± 0.0023)
Test AUC    : 0.7733
Test recall : 0.7005
Test cost   : 31296

Entraînement : RandomForest
Cross-validation 5 folds...
Entraînement final sur le train complet...


2026/05/11 17:52:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/11 17:52:19 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


CV AUC      : 0.7244 (± 0.0031)
Test AUC    : 0.7293
Test recall : 0.0022
Test cost   : 49548

Entraînement : LightGBM
Cross-validation 5 folds...
Entraînement final sur le train complet...


2026/05/11 17:55:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/11 17:55:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


CV AUC      : 0.7815 (± 0.0011)
Test AUC    : 0.7846
Test recall : 0.7055
Test cost   : 30028

Entraînement : XGBoost
Cross-validation 5 folds...
Entraînement final sur le train complet...


2026/05/11 17:59:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/11 17:59:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


CV AUC      : 0.7610 (± 0.0023)
Test AUC    : 0.7668
Test recall : 0.6058
Test cost   : 31819


## 8. Comparaison des résultats

Tableau récapitulatif des 3 modèles. Les mêmes données sont disponibles dans l'UI MLflow avec plus de détails.

In [8]:
results_df = pd.DataFrame(results).T
results_df = results_df[['cv_auc_mean', 'cv_auc_std', 'test_auc', 
                         'test_recall', 'test_precision', 'test_f1',
                         'test_business_cost', 'cv_business_cost_mean']]
results_df.round(4)

,cv_auc_mean,cv_auc_std,test_auc,test_recall,test_precision,test_f1,test_business_cost,cv_business_cost_mean
LogisticRegression,0.7708,0.0023,0.7733,0.7005,0.1747,0.2797,31296.0,25134.6
RandomForest,0.7244,0.0031,0.7293,0.0022,0.5789,0.0044,49548.0,39659.4
LightGBM,0.7815,0.0011,0.7846,0.7055,0.1852,0.2934,30028.0,24479.8
XGBoost,0.7610,0.0023,0.7668,0.6058,0.1972,0.2975,31819.0,26020.0


**Observations :**

- **LightGBM** sort en tête : meilleure AUC en CV (0.78), meilleur recall (0.71) et coût métier le plus bas. Cohérent avec ce qu'on attend du boosting sur ce type de données.
- **XGBoost** donne des résultats un peu en dessous (AUC test 0.77, recall 0.61, coût 31 819). C'est un peu surprenant car XGBoost et LightGBM sont deux implémentations du gradient boosting très proches. Avec les hyperparamètres par défaut, LightGBM semble mieux calibré pour ce type de données déséquilibrées. À voir si XGBoost rattrape son retard après optimisation à l'étape suivante.
- **LogisticRegression** s'en sort bien (AUC test 0.77, recall 0.70). C'est rassurant : ça veut dire que les features sont déjà bien construites au notebook 02 et qu'un modèle linéaire arrive à capter pas mal du signal.
- **RandomForest** déçoit : recall quasi nul (0.002). Avec ses hyperparamètres par défaut (arbres profonds, 100 estimateurs), il overfit la classe majoritaire malgré `class_weight='balanced'`. On verra à l'étape suivante si une optimisation des hyperparamètres (profondeur limitée, plus d'estimateurs) améliore les choses.
- **L'écart-type CV** est très faible pour les trois modèles à arbres (~0.001-0.003) → métriques stables, donc fiables.

À l'étape suivante : optimiser les hyperparamètres (notamment de RF et XGBoost) et **optimiser le seuil de décision** pour minimiser le coût métier (le seuil 0.5 par défaut n'est jamais optimal sur un problème déséquilibré avec coûts asymétriques).

## 9. Visualisation dans l'UI MLflow

Pour visualiser les runs dans l'interface web MLflow :

1. Ouvrir un terminal dans le dossier du projet (`P6_credit_scoring/`)
2. Lancer la commande :

```bash
uv run mlflow ui
```

3. Ouvrir le navigateur à l'adresse : http://localhost:5000

Dans l'UI, on peut :
- Voir tous les runs de l'experiment `P6_credit_scoring`
- Comparer les métriques côte à côte (cocher plusieurs runs puis cliquer sur "Compare")
- Filtrer par tags (ex. `tags.model_family = "LightGBM"`)
- Télécharger les modèles sérialisés

C'est cette infrastructure de tracking qu'on réutilisera à l'étape 4 quand on optimisera les hyperparamètres et le seuil métier.